# TDAH and MI result generation

Set `DEBUG = True` in the setup cell to write CSV outputs under `Temp/Results`.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/STF-KernelSHAP.git"
REPO_NAME = "STF-KernelSHAP"
PACKAGE_PATH = Path("src") / "stf_kernelshap"


def run_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        message = [
            f"Command failed: {' '.join(map(str, command))}",
            f"Return code: {result.returncode}",
        ]
        if result.stdout:
            message.append(f"STDOUT:\n{result.stdout}")
        if result.stderr:
            message.append(f"STDERR:\n{result.stderr}")
        raise RuntimeError("\n".join(message))
    return result


def resolve_working_root():
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    if Path("/content").exists():
        return Path("/content")
    return Path.cwd().resolve()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / PACKAGE_PATH).exists():
            return candidate

    working_root = resolve_working_root()
    clone_dir = working_root / REPO_NAME

    if (clone_dir / PACKAGE_PATH).exists():
        return clone_dir.resolve()

    if clone_dir.exists() and (clone_dir / ".git").exists():
        run_command(["git", "-C", str(clone_dir), "pull", "--ff-only"])
        if (clone_dir / PACKAGE_PATH).exists():
            return clone_dir.resolve()

    if clone_dir.exists() and not (clone_dir / PACKAGE_PATH).exists():
        clone_dir = working_root / f"{REPO_NAME}_repo"

    if not clone_dir.exists():
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])

    if not (clone_dir / PACKAGE_PATH).exists():
        raise FileNotFoundError(
            f"El repositorio clonado no contiene {PACKAGE_PATH}: {clone_dir}"
        )

    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    run_command([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ])


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

RUNNING_IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
RUNNING_IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
INSTALL_REQUIREMENTS = RUNNING_IN_COLAB or RUNNING_IN_KAGGLE

if INSTALL_REQUIREMENTS:
    install_project_requirements(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INSTALL_REQUIREMENTS:", INSTALL_REQUIREMENTS)


PROJECT_ROOT: /Users/diego/Documents/STF-KernelSHAP
INSTALL_REQUIREMENTS: False


In [2]:
from stf_kernelshap.notebook_setup import setup_notebook_environment

DEBUG = True
paths = setup_notebook_environment(debug=DEBUG)

DATA_DIR = paths.data_dir
MODELS_DIR = paths.models_dir
REPO_MODELS_DIR = paths.repo_models_dir
REPO_RESULTS_DIR = paths.repo_results_dir
REPO_FIGURES_DIR = paths.repo_figures_dir
OUTPUT_MODELS_DIR = paths.output_models_dir
RESULTS_DIR = paths.results_dir
FIGURES_DIR = paths.figures_dir


Repository root: /Users/diego/Documents/STF-KernelSHAP
Dataset root: /Users/diego/Documents/STF-KernelSHAP/MI_TDAH_Dataset
Debug mode: True
Results dir: /Users/diego/Documents/STF-KernelSHAP/Temp/Results
Figures dir: /Users/diego/Documents/STF-KernelSHAP/Temp/Figures
Output models dir: /Users/diego/Documents/STF-KernelSHAP/Temp/Models


In [3]:
import pickle

from stf_kernelshap.evaluation.pipeline import (
    evaluate_all_windows_to_single_csv,
    train_tdah_with_best_optuna_config,
)
from stf_kernelshap.data import get_segmented_data


In [4]:
model_name = "shallowconvnet"  # "eegnet", "shallowconvnet", "tgarnet"
folds_path = DATA_DIR / "TDAH" / "folds.pkl"
journal_file = REPO_MODELS_DIR / "TDAH" / "Optuna" / f"study_{model_name}_TDAH.journal"
study_name = f"study_{model_name}_TDAH"

base_model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
}


In [5]:
with open(folds_path, "rb") as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data(
    path_adhd=str(DATA_DIR / "TDAH" / "ieee" / "ADHD_group"),
    path_control=str(DATA_DIR / "TDAH" / "ieee" / "Control_group"),
)


In [6]:
results = train_tdah_with_best_optuna_config(
    model_name=model_name,
    study_name=study_name,
    journal_file=str(journal_file),
    base_model_args=base_model_args,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    epochs=2,
    batch_size=16,
    seed=34,
    seed_gap=100,
    n_repeats=2,
)

df_predictions = results["subject_predictions_df"]
tdah_output_csv = RESULTS_DIR / "TDAH_results.csv"
#df_predictions.to_csv(tdah_output_csv, index=False)
df_predictions


DEBUG: Intentando cargar el estudio 'study_shallowconvnet_TDAH' del journal '/Users/diego/Documents/STF-KernelSHAP/Models/TDAH/Optuna/study_shallowconvnet_TDAH.journal'
DEBUG: Journal file '/Users/diego/Documents/STF-KernelSHAP/Models/TDAH/Optuna/study_shallowconvnet_TDAH.journal' existe. Tamaño: 61570 bytes.
DEBUG: Estudio 'study_shallowconvnet_TDAH' cargado exitosamente. Número de trials: 20
Best trial: 16
Best value: 0.7018126050528857
Best params: {'pool_size': 64, 'pool_stride': 2, 'n_filters': 24, 'kernel_length': 40, 'dropoutRate': 0.6500000000000001, 'norm_rate': 0.40000000000000013, 'learning_rate': 0.000567151675173216}


2026-06-25 02:31:46.969266: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-06-25 02:32:03.118006: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

,model,repeat,fold,model_seed,subject,n_segments,true_class,pred_majority_class,segment_accuracy,prob_class_0,prob_class_1
0,shallowconvnet,0,0,34,v107,76,0,1,0.144737,0.150727,0.849273
1,shallowconvnet,0,0,34,v108,73,0,1,0.191781,0.200876,0.799124
2,shallowconvnet,0,0,34,v121,62,0,1,0.016129,0.015600,0.984400
3,shallowconvnet,0,0,34,v127,56,0,1,0.107143,0.111519,0.888481
4,shallowconvnet,0,0,34,v129,42,0,1,0.357143,0.364950,0.635050
...,...,...,...,...,...,...,...,...,...,...,...
233,shallowconvnet,1,4,10434,v307,88,0,1,0.102273,0.096615,0.903385
234,shallowconvnet,1,4,10434,v34p,75,1,1,0.546667,0.458165,0.541835
235,shallowconvnet,1,4,10434,v49p,64,0,1,0.203125,0.207846,0.792154
236,shallowconvnet,1,4,10434,v53p,73,0,0,0.767123,0.776649,0.223351


In [ ]:
df_results = evaluate_all_windows_to_single_csv(
    windows=("2.5-5", "0-7"),
    models=("eegnet", "shallowconvnet", "tgarnet"),
    subjects=None,
    data_mi_root=str(DATA_DIR / "MI"),
    models_mi_root=str(REPO_MODELS_DIR / "MI"),
    output_csv=str(RESULTS_DIR / "MI_results.csv"),
)
df_results



EVALUANDO VENTANA: 2.5-5

Modelo: eegnet
Sujetos: [1]

Modelo: shallowconvnet
Sujetos: [1]

Modelo: tgarnet
Sujetos: [1]

EVALUANDO VENTANA: 0-7

Modelo: eegnet
Sujetos: [1]

Modelo: shallowconvnet
Sujetos: [1]

Modelo: tgarnet
Sujetos: [1]


,window,model,subject,n_folds,seed,mean_accuracy,std_accuracy,mean_recall,std_recall,mean_precision,std_precision,mean_kappa,std_kappa,mean_auc,std_auc
0,2.5-5,eegnet,1,5,42,0.845,0.059687,0.845,0.059687,0.856361,0.054674,0.69,0.119373,0.9415,0.028096
1,2.5-5,shallowconvnet,1,5,42,0.840,0.051841,0.840,0.051841,0.851726,0.054438,0.68,0.103682,0.9110,0.061811
2,2.5-5,tgarnet,1,5,42,0.765,0.022361,0.765,0.022361,0.775333,0.026420,0.53,0.044721,0.8585,0.054589
3,0-7,eegnet,1,5,42,0.845,0.032596,0.845,0.032596,0.848862,0.030434,0.69,0.065192,0.9310,0.019969
4,0-7,shallowconvnet,1,5,42,0.865,0.037914,0.865,0.037914,0.868463,0.037753,0.73,0.075829,0.9415,0.024405
5,0-7,tgarnet,1,5,42,0.715,0.041833,0.715,0.041833,0.723942,0.037270,0.43,0.083666,0.7695,0.040288
